In [7]:
# lightning_mobilenet_cifar_mlflow.py
import argparse, os, torch, pytorch_lightning as pl
from torch import nn
from torch.utils.data import random_split, DataLoader
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torchvision.models import mobilenet_v3_small
import mlflow
import torch.nn.init as init
import torchmetrics
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from dataclasses import dataclass


In [8]:
# ------------- Data ----------------
class CIFAR10DataModule(pl.LightningDataModule):
    def __init__(self, data_dir="./data", batch_size=128, num_workers=4):
        super().__init__()
        self.save_hyperparameters()
        self.train_t = T.Compose([
            T.Resize(224), T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize((0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)),
        ])
        self.test_t = T.Compose([
            T.Resize(224), T.ToTensor(),
            T.Normalize((0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)),
        ])

    def prepare_data(self):
        CIFAR10(self.hparams.data_dir, train=True, download=False)
        CIFAR10(self.hparams.data_dir, train=False, download=False)

    def setup(self, stage=None):
        full = CIFAR10(self.hparams.data_dir, train=True, transform=self.train_t)
        self.train_set, self.val_set = random_split(full, [45000, 5000])
        self.test_set = CIFAR10(self.hparams.data_dir, train=False, transform=self.test_t)

    def _loader(self, ds, shuffle):
        return DataLoader(ds, batch_size=self.hparams.batch_size,
                          shuffle=shuffle, num_workers=self.hparams.num_workers,
                          pin_memory=True)

    def train_dataloader(self): return self._loader(self.train_set, True)
    def val_dataloader(self):   return self._loader(self.val_set, False)
    def test_dataloader(self):  return self._loader(self.test_set, False)

In [9]:
# ---------------------Base Model--------------------------------
class MobileNetCIFAR(pl.LightningModule):
    def __init__(self, num_classes=10, lr=3e-4):
        super().__init__()
        # self.save_hyperparameters()

        backbone = mobilenet_v3_small(weights="DEFAULT")
        # if freeze_backbone:
        #     for p in backbone.parameters():
        #         p.requires_grad = False

        in_features = backbone.classifier[3].in_features
        backbone.classifier[3] = nn.Linear(in_features, num_classes)
        self.model = backbone

        # self.criterion = nn.CrossEntropyLoss()
        # self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        # self.val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        # self.test_acc  = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x): 
        return self.model(x)

# Load a pretrained model
basemodel = MobileNetCIFAR.load_from_checkpoint('MEED/mlruns/base_model_mobilenet/f9230b854f824153814a2d4a407b4157/checkpoints/epoch=16-step=5984.ckpt').eval();


In [11]:
class JSD(nn.Module):
    def __init__(self):
        super(JSD, self).__init__()
        self.kl = nn.KLDivLoss(reduction='batchmean', log_target=True)

    def forward(self, p: torch.tensor, q: torch.tensor):
        p, q = p.view(-1, p.size(-1)), q.view(-1, q.size(-1))
        m = (0.5 * (p + q)).log()
        return 0.5 * (self.kl(m, p.log()) + self.kl(m, q.log()))

@dataclass(slots=True)
class ModelCfg:
    input_dim:   int   = 3072
    hidden_dim:  int   = 512
    output_dim:  int   = 209
    num_heads:   int   = 8        # hidden_dim must be divisible by num_heads
    dropout:     float = 0.10
    pool:        str   = "mean"   # "mean" or "cls"
    device:      str   = "cuda" if torch.cuda.is_available() else "cpu"
    amp:         bool  = True     # Automatic Mixed Precision
    lr:          float = 3e-4


# --------------------------------------------------------------------------- #
# Model definition                                                            #
# --------------------------------------------------------------------------- #
class TinyNet(pl.LightningModule):
    """
    Accepts (B, D) or (B, L, D) and returns logits (B, 209).
    Apply sigmoid outside the model when probabilities are needed.
    """
    def __init__(self, cfg: ModelCfg = ModelCfg()) -> None:
        super().__init__()
        self.save_hyperparameters()
        if cfg.hidden_dim % cfg.num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.cfg   = cfg
        self.fc1   = nn.Linear(cfg.input_dim, cfg.hidden_dim)
        self.act   = nn.ReLU()

        self.attn  = nn.MultiheadAttention(
            embed_dim   = cfg.hidden_dim,
            num_heads   = cfg.num_heads,
            dropout     = cfg.dropout,
            batch_first = True,
        )
        self.norm  = nn.LayerNorm(cfg.hidden_dim)
        self.fc_out = nn.Linear(cfg.hidden_dim, cfg.output_dim)

        # self._init_params()
        self.basemodel = basemodel
        self.criterion = JSD()#nn.CrossEntropyLoss()
        self.clf_loss  = nn.CrossEntropyLoss()
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc  = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        
    # --------------------------------------------------------------------- #
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Ensure sequence dimension
        if x.dim() == 2:                       # (B, D) ➜ (B, 1, D)
            x = x.unsqueeze(1)

        h = self.act(self.fc1(x))              # (B, L, 512)

        attn_out, _ = self.attn(h, h, h)       # MHA
        h = self.norm(h + attn_out)            # Residual + LN
        h = self.act(h)

        # Sequence pooling
        h = h.mean(dim=1) if self.cfg.pool == "mean" else h[:, 0, :]

        return self.fc_out(h)    
        
    # ---------------------------------------------------------
    def _init_params(self):
        for m in self.modules():
    
            # 1) Convs & Linears  ──────────────────────────────────────────
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    init.zeros_(m.bias)
    
            # 2) Multi-head Self-/Cross-Attention  ─────────────────────────
            elif isinstance(m, nn.MultiheadAttention):
                # Query-Key-Value projection weights share one big matrix:
                init.xavier_uniform_(m.in_proj_weight)
                if m.in_proj_bias is not None:
                    init.zeros_(m.in_proj_bias)

                # Output projection
                init.xavier_uniform_(m.out_proj.weight)
                if m.out_proj.bias is not None:
                    init.zeros_(m.out_proj.bias)

            # 3) (Optional) Embeddings, LayerNorm, etc.  ───────────────────
            elif isinstance(m, nn.Embedding):
                init.normal_(m.weight, mean=0.0, std=0.02)
            
            elif isinstance(m, nn.LayerNorm):
                init.ones_(m.weight)
                init.zeros_(m.bias)

    # def forward(self, x): return self.model(x)
    # ------------- 2. FREQUENCY UTILITIES ----------------------------------------
    def fft_features(self, x: torch.Tensor) -> torch.Tensor:
        """
        Raw frequency features: log(1+|FFTshift(FFT(x))|) flattened per image.
        x: (B,C,H,W) in [0,1]
        returns (B, C*H*W)
        """
        # complex FFT per channel
        f = torch.fft.fft2(x, norm="ortho")                    # (B,C,H,W)
        f = torch.fft.fftshift(f, dim=(-2, -1))
        mag = torch.log1p(torch.abs(f))                        # log-magnitude
        return mag.flatten(start_dim=1)                        # (B, C*H*W)

    def signed_to_binary(self, output, threshold=0.5):
        """
        Converts the signed output of a multi-label model to a binary vector.
        
        Args:
        output (torch.Tensor): The output tensor from the model (containing signed values).
        threshold (float, optional): The threshold for binarization. Defaults to 0.5.
        
        Returns:
        torch.Tensor: A binary tensor where 1 indicates the class is present and 0 indicates it's absent.
        """
        binary_output = (output >= threshold).int()
        return binary_output
        
    def prune_by_mask(self, model: nn.Module, mask: torch.Tensor) -> nn.Module:
        """
        Replace the k-th sub-module (depth-first, exclusive of the root)
        with nn.Identity() whenever mask[k] == 0.
    
        The traversal order is the same as list(model.named_modules())[1:].
        """
        # Collect (path, module, parent) triples in depth-first order
        triplets = []
        for name, module in list(model.named_modules())[1:]:           # skip root
            parent_path = ".".join(name.split(".")[:-1])
            child_name  = name.split(".")[-1]
            parent = model.get_submodule(parent_path) if parent_path else model
            triplets.append((child_name, module, parent))
    
        assert len(mask) == len(triplets), (
            f"Mask length {len(mask)} must equal #layers {len(triplets)}"
        )
    
        # Swap out the pruned layers
        for keep, (child_name, _, parent) in zip(mask, triplets):
            if keep.item() == 0:
                setattr(parent, child_name, nn.Identity())
    
        return model

    def _shared_step(self, batch, stage):
        x, y = batch
        feat_x = fft_features(x)
        mask = self(feat_x)
        b_mask = signed_to_binary(mask)
        pruned_basemodel = prune_by_mask(basemodel, b_mask)

        with no_grad():
            unp_logits = basemodel(x)
            p_logits = pruned_basemodel(x)     
        
        # logits, _ = self(x)
        loss   = self.criterion(unp_logits, p_logits)
        cl_loss = self.clf_loss(p_logits, y)
        p_preds  = p_logits.argmax(1)
        getattr(self, f"{stage}_acc")(p_preds, y)
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True)
        self.log(f"{stage}_clf_loss", cl_loss, prog_bar=True, on_epoch=True)
        self.log(f"{stage}_acc", getattr(self, f"{stage}_acc"), prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, b, i):  return self._shared_step(b, "train")
    def validation_step(self, b, i): self._shared_step(b, "val")
    def test_step(self, b, i):       self._shared_step(b, "test")

    def configure_optimizers(self):
        opt = torch.optim.RMSprop(
            self.parameters(),
            lr=self.hparams.lr,
            alpha=0.99, eps=1e-06, 
            weight_decay=5e-4, momentum=0.9, 
            centered = False
        ) 
        # torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
        return [opt], [sch]


# checkpoint_callback = ModelCheckpoint(
#     dirpath="checkpoints",
#     filename="{epoch}-{val_clf_loss:.2f}",
#     save_top_k=1,
#     monitor="val_clf_loss",
# )




usage: ipykernel_launcher.py [-h] [--batch_size BATCH_SIZE] [--lr LR]
                             [--max_epochs MAX_EPOCHS]
                             [--freeze_backbone FREEZE_BACKBONE]
                             [--tracking_uri TRACKING_URI]
ipykernel_launcher.py: error: unrecognized arguments: -f /users/aupendrannair/.local/share/jupyter/runtime/kernel-5e276944-9acb-4949-949f-5476768dec70.json


SystemExit: 2

/users/aupendrannair/.conda/envs/meed/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ------------- Script --------------
def cli_main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=3e-5)
    parser.add_argument("--max_epochs", type=int, default=50)
    parser.add_argument("--freeze_backbone", type=bool, default=False)
    parser.add_argument("--tracking_uri", type=str, default="file:./mlruns/tinynet")
    args = parser.parse_args()

    pl.seed_everything(42, workers=True)

    # MLflow logger (creates experiment on first use)
    mlf_logger = MLFlowLogger(
        experiment_name="cifar10_train_TINYNET",
        tracking_uri=args.tracking_uri,
        log_model=True,                           # auto-logs best checkpoint
        tags={"model": "train_tinynet"}
    )

    dm = CIFAR10DataModule(batch_size=args.batch_size)
    model = TinyNetCIFAR(lr=args.lr)

    ckpt_cb = ModelCheckpoint(
            dirpath="checkpoints",
            filename="{epoch}-{val_clf_loss:.2f}",
            save_top_k=1,
            monitor="val_clf_loss",
            mode = "min"
        )

    trainer = pl.Trainer(
        max_epochs=args.max_epochs,
        accelerator="auto",
        devices = 4,
        strategy = 'ddp',
        precision="bf16-mixed" if torch.cuda.is_available() else "32-true",
        logger=mlf_logger,
        callbacks=[ckpt_cb],
        log_every_n_steps=25
    )

    trainer.fit(model, dm, ckpt_path = None)
    trainer.test(model, datamodule=dm)
    
if __name__ == "__main__":
    cli_main()